In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

#leyendo base de datos
df = pd.read_csv('Book_bomberos_2015_a_sep_de_2025_ok_pa.csv',sep=";")

regiones_2 = ["anho","Región de Arica","Región de Tarapacá","Región de Antofagasta","Región de Atacama","Región de Coquimbo","Región de Valparaíso","Región Metropolitana","Región del Libertador Bernardo O'higgins","Región del Maule","Región del Ñuble","Región del Bío Bío","Región de la Araucanía","Región de los Ríos","Región de los Lagos","Región de Aysén del General Carlos Ibáñez del Campo","Región de Magallanes y de la Antártica Chilena"]
df = pd.read_csv('Book_bomberos_2015_a_sep_de_2025_ok_pa.csv',sep=";")
meses_a_numero = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4,
    "mayo": 5, "junio": 6, "julio": 7, "agosto": 8,
    "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12
}

df["mes_num"] = df["mes"].str.lower().map(meses_a_numero)

df_totales_anuales = df.groupby("anho").sum(numeric_only=True).reset_index() 

df.head()

,Unnamed: 0,tipo de incendio,año,mes,anho,Región de Arica,Región de Tarapacá,Región de Antofagasta,Región de Atacama,Región de Coquimbo,...,Región del Maule,Región del Ñuble,Región del Bío Bío,Región de la Araucanía,Región de los Ríos,Región de los Lagos,Región de Aysén del General Carlos Ibáñez del Campo,Región de Magallanes y de la Antártica Chilena,Chile,mes_num
0,0,Estructural 10-0,2015,enero,1,9,23,27,10,16,...,82,30,99,88,42,99,7,20,899,1
1,1,Vehículos 10-1,2015,enero,1,3,6,5,4,12,...,21,10,18,19,9,17,1,4,341,1
2,2,Pastizales y/o Basura 10-2,2015,enero,1,18,24,47,59,44,...,695,263,846,756,247,281,5,18,5419,1
3,3,Rescate de Emergencia 10-3,2015,enero,1,4,12,15,5,11,...,59,20,91,45,33,50,7,2,678,1
4,4,Rescate Vehicular 10-4,2015,enero,1,14,35,33,54,42,...,106,52,157,105,46,76,7,9,1330,1


In [12]:
import os
import sys
from contextlib import contextmanager
from pycaret.regression import *

# 1. Definimos una función para silenciar todo
@contextmanager
def suppress_output():
    with open(os.devnull, 'w') as fnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        try:
            sys.stdout = fnull
            sys.stderr = fnull
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

# Tu preparación de datos
a = 15
# ... (tu código de carga de datos y groupby) ...
# Suponiendo que df_totales_anuales ya está creado
data_py = df_totales_anuales[["año", "mes_num", "Chile"]].copy()

# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Chile", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Chile
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
xgboost,Extreme Gradient Boosting,939.2464,1624983.2137,1174.2888,0.7341,0.1029,0.0827,0.5407
gbr,Gradient Boosting Regressor,936.4356,1646240.2790,1195.7594,0.7135,0.1025,0.0822,0.0587
rf,Random Forest Regressor,980.2563,1882530.0668,1233.1730,0.6966,0.1057,0.0874,0.1707
et,Extra Trees Regressor,991.9127,1866795.6329,1243.0813,0.6957,0.1075,0.0875,0.1260
dt,Decision Tree Regressor,1244.9095,2760860.6397,1534.6359,0.5585,0.1258,0.1049,0.0133
lightgbm,Light Gradient Boosting Machine,1215.8113,2943536.0872,1558.6459,0.5347,0.1321,0.1081,0.2027
knn,K Neighbors Regressor,1206.2022,2859645.2172,1546.0091,0.5075,0.1306,0.1074,0.0387
ada,AdaBoost Regressor,1296.8601,2789827.6796,1597.2113,0.4554,0.1346,0.1148,0.0673
br,Bayesian Ridge,1560.1024,4254365.7741,1962.0224,0.2330,0.1661,0.1386,0.0140
llar,Lasso Least Angle Regression,1555.9786,4253320.2810,1961.0221,0.2329,0.1658,0.1381,0.0133


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extreme Gradient Boosting,955.0502,1822079.7930,1349.8444,0.7895,0.1056,0.0770


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,1008.9556,1529135.4228,1236.5822,0.7509,0.0994,0.0829
1,1108.7628,1912875.7404,1383.0675,0.8558,0.1008,0.0837
2,629.0910,634496.3210,796.5528,0.8892,0.0552,0.0455
3,749.0967,817597.4626,904.2110,0.5166,0.0715,0.0606
4,863.7495,1265686.0560,1125.0271,0.6909,0.0897,0.0661
5,1907.0816,10167496.9569,3188.6513,-0.1854,0.3724,0.2910
6,747.1139,638805.2435,799.2529,0.8348,0.0726,0.0695
7,669.9392,598180.4738,773.4213,0.8790,0.0703,0.0604
8,623.3184,476949.1219,690.6150,0.7145,0.0643,0.0600


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extreme Gradient Boosting,955.0502,1822079.7930,1349.8444,0.7895,0.1056,0.0770


¡Proceso terminado!
XGBRegressor(base_score=None, booster='gbtree', callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device='gpu', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=-1, num_parallel_tree=None, ...)


Aquí tienes un análisis técnico conciso tipo informe:El modelado predictivo para la variable "Chile", basado en 129 registros históricos, identificó a Extreme Gradient Boosting (XGBoost) como el algoritmo óptimo, superando significativamente a los modelos lineales ($R^2 \approx 0.23$) y confirmando la naturaleza no lineal de la serie. Tras la optimización de hiperparámetros validada mediante 15 folds, el modelo alcanzó un $R^2$ promedio de 0.67 y un MAPE del 9.03%, lo que implica que las predicciones tienen, en promedio, un margen de error inferior al 10%. No obstante, se detecta una alta volatilidad en la estabilidad del modelo (Std 0.25), evidenciada críticamente en el Fold 5 ($R^2$ de -0.18), lo que sugiere una anomalía estructural o "quiebre" en los datos durante ese periodo específico. En conclusión, el modelo es robusto para la mayoría de los escenarios.

In [13]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num", "Región de Arica"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Arica", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Arica")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Arica
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,9.5679,145.9242,11.5605,0.5105,0.1642,0.1451,0.2013
et,Extra Trees Regressor,9.7990,159.5915,12.0551,0.4511,0.1762,0.1502,0.1227
ada,AdaBoost Regressor,9.8961,148.5717,11.9502,0.4435,0.1711,0.1507,0.0480
rf,Random Forest Regressor,10.1396,162.1342,12.2673,0.4284,0.1752,0.1527,0.1680
gbr,Gradient Boosting Regressor,10.2468,164.3610,12.3828,0.3738,0.1790,0.1551,0.0593
xgboost,Extreme Gradient Boosting,10.5033,185.1522,13.2342,0.3334,0.1915,0.1578,0.4073
knn,K Neighbors Regressor,11.2879,189.0458,13.4879,0.3286,0.1855,0.1638,0.0373
dt,Decision Tree Regressor,11.6302,226.1381,14.4036,0.1329,0.2070,0.1738,0.0147
omp,Orthogonal Matching Pursuit,14.4029,344.2158,17.9748,-0.0855,0.2497,0.2184,0.0120
dummy,Dummy Regressor,14.4132,344.1240,18.0507,-0.0976,0.2516,0.2197,0.0113


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,7.1252,87.4325,9.3505,0.7247,0.1275,0.0986


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,6.7423,64.6777,8.0422,0.7708,0.1201,0.1011
1,12.6155,206.6023,14.3737,0.6625,0.1702,0.1544
2,8.5585,108.2392,10.4038,0.6069,0.1671,0.1426
3,11.4255,149.0013,12.2066,0.0851,0.1538,0.1550
4,10.0266,139.9111,11.8284,0.3335,0.1719,0.1410
5,13.8945,290.5476,17.0455,0.4299,0.3531,0.3343
6,10.5324,121.6871,11.0312,0.5337,0.1843,0.1740
7,8.2350,106.2033,10.3055,0.7011,0.1424,0.1106
8,12.3119,215.0141,14.6634,-0.0289,0.1770,0.1683


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,7.1252,87.4325,9.3505,0.7247,0.1275,0.0986


¡Proceso terminado!
LGBMRegressor(device='gpu', n_jobs=-1, random_state=123)
Proceso finalizado para Región de Arica


El análisis predictivo para la "Región de Arica" identificó al algoritmo LightGBM como el modelo más eficiente, superando a Extra Trees y Random Forest al capturar mejor la no linealidad de los datos frente al nulo desempeño de los modelos lineales. Tras la validación cruzada, el modelo arroja un MAPE promedio del 15.90%, indicando un margen de error moderado, pero un $R^2$ promedio bajo de 0.37. Lo más crítico es la alta inestabilidad temporal detectada: mientras que en periodos como el Fold 0 el ajuste es excelente ($R^2$ 0.77), en otros como el Fold 11 el modelo colapsa ($R^2$ -0.41). Esta volatilidad sugiere que la serie de Arica presenta cambios estructurales abruptos o datos atípicos que impiden una generalización robusta.

In [14]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num", "Región de Tarapacá"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Tarapacá", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Tarapacá")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Tarapacá
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,24.8777,962.1441,29.9776,0.0750,0.2221,0.1893,0.1867
gbr,Gradient Boosting Regressor,26.2624,1000.1375,30.8249,0.0520,0.2234,0.1950,0.0600
ada,AdaBoost Regressor,25.9966,1016.6523,31.1866,0.0465,0.2320,0.2044,0.0727
et,Extra Trees Regressor,26.4269,1051.8536,31.4872,0.0060,0.2286,0.1982,0.1400
knn,K Neighbors Regressor,26.9117,1104.8248,32.3140,-0.0265,0.2469,0.2166,0.0407
xgboost,Extreme Gradient Boosting,28.5762,1249.5758,34.3818,-0.2665,0.2444,0.2149,0.4353
omp,Orthogonal Matching Pursuit,31.9908,1696.6551,39.7522,-0.2955,0.2977,0.2748,0.0133
br,Bayesian Ridge,32.1474,1704.0080,39.8433,-0.2970,0.2981,0.2755,0.0133
lasso,Lasso Regression,32.3954,1741.0312,40.3204,-0.3388,0.3018,0.2779,0.0120
llar,Lasso Least Angle Regression,32.3954,1741.0312,40.3204,-0.3388,0.3018,0.2779,0.0153


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,23.4365,789.5382,28.0987,0.0505,0.1955,0.1696


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,28.7006,956.7945,30.9321,0.4043,0.2074,0.2018
1,41.5693,2648.3718,51.4623,0.1572,0.2741,0.2189
2,36.0475,2052.3607,45.3030,-0.6050,0.2588,0.1909
3,19.6292,602.6849,24.5496,0.6187,0.1501,0.1232
4,18.3580,531.6654,23.0579,0.4325,0.1895,0.1627
5,29.3428,1587.3854,39.8420,0.3844,0.5123,0.5100
6,25.2988,987.1396,31.4188,-0.0265,0.1952,0.1456
7,13.7308,376.5731,19.4055,0.2282,0.1400,0.1062
8,21.4119,650.4218,25.5034,-0.1657,0.2129,0.2006


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,20.5404,605.8671,24.6144,0.2714,0.1751,0.1509


¡Proceso terminado!
RandomForestRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región de Tarapacá


El ejercicio de modelado para la Región de Tarapacá determinó que el Random Forest Regressor es el estimador más estable dentro de un conjunto de bajo rendimiento general, superando marginalmente a modelos de boosting como XGBoost. Sin embargo, la capacidad predictiva es débil: el modelo alcanza un $R^2$ promedio de apenas 0.09, lo que implica que apenas logra explicar la variabilidad de los datos, acompañado de un MAPE del 20.76%, un margen de error considerable para fines operativos. El análisis de validación cruzada revela una inestabilidad crítica: mientras que en el Fold 3 el modelo logra capturar la tendencia ($R^2$ 0.61), en segmentos como el Fold 9 y Fold 13 el ajuste colapsa con valores negativos severos. Esto sugiere que la serie de Tarapacá contiene ruido excesivo o determinantes exógenos que las variables actuales no logran capturar, haciendo que el modelo actual no sea confiable sin una reingeniería de características.

In [16]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num", "Región de Antofagasta"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Antofagasta", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Antofagasta")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Antofagasta
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,32.0334,1952.8803,41.3423,0.2772,0.1491,0.1188,0.1813
ada,AdaBoost Regressor,32.3931,2122.7812,42.9274,0.2365,0.1568,0.1231,0.0580
lightgbm,Light Gradient Boosting Machine,33.5934,1944.4581,41.5490,0.2305,0.1491,0.1257,0.2087
gbr,Gradient Boosting Regressor,32.6353,2093.4405,42.3304,0.2205,0.1562,0.1233,0.0647
knn,K Neighbors Regressor,35.7670,2117.2895,43.6357,0.1700,0.1529,0.1309,0.0393
et,Extra Trees Regressor,34.6303,2185.2702,43.9236,0.1666,0.1618,0.1312,0.1280
xgboost,Extreme Gradient Boosting,35.6802,2392.7879,45.2151,0.1038,0.1698,0.1375,0.4113
br,Bayesian Ridge,37.4848,2597.1468,47.2402,0.0791,0.1665,0.1409,0.0147
omp,Orthogonal Matching Pursuit,37.3675,2589.2356,47.2321,0.0719,0.1662,0.1401,0.0207
en,Elastic Net,37.6218,2624.3370,47.5037,0.0597,0.1673,0.1412,0.0127


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,22.7558,804.7886,28.3688,0.7914,0.1142,0.0884


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,13.0761,431.6505,20.7762,0.5030,0.0745,0.0429
1,38.4641,2975.2971,54.5463,0.5035,0.1572,0.1176
2,16.3260,481.5515,21.9443,0.6543,0.0853,0.0576
3,21.3083,1032.3684,32.1305,0.1148,0.0937,0.0629
4,19.0450,478.3834,21.8720,0.6200,0.0699,0.0626
5,51.1540,5576.1687,74.6737,0.0177,0.3545,0.3082
6,36.4189,1858.1128,43.1058,0.6324,0.1367,0.1225
7,31.8413,1599.5916,39.9949,0.6469,0.1347,0.1172
8,40.3194,2262.1647,47.5622,0.6069,0.2275,0.1970


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,27.8308,1197.4729,34.6045,0.6896,0.1488,0.1140


¡Proceso terminado!
RandomForestRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región de Antofagasta


El modelado para la Región de Antofagasta presenta un escenario más optimista que los anteriores, seleccionando al Random Forest Regressor como el estimador más robusto frente a modelos lineales ineficaces. Aunque el $R^2$ promedio de validación cruzada es moderado (0.35), el modelo exhibe una excelente capacidad de generalización en el conjunto de prueba final, alcanzando un $R^2$ de 0.69. Operativamente es funcional, con un MAPE promedio del 11.70%, lo que implica un error de pronóstico bajo y aceptable para la toma de decisiones. No obstante, la volatilidad persiste: existe una discrepancia masiva entre periodos estables (como el Fold 7 con $R^2$ 0.65) y quiebres estructurales severos (como el Fold 14 con $R^2$ negativo), lo que sugiere que, aunque el modelo es útil, la región está sujeta a shocks puntuales que el algoritmo no logra anticipar sin variables externas.

In [17]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num", "Región de Atacama"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Atacama", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Atacama")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Atacama
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,23.0753,933.6091,27.7683,-0.0491,0.1625,0.1432,0.1367
knn,K Neighbors Regressor,23.3067,909.2044,28.1021,-0.0723,0.1617,0.1427,0.0380
lightgbm,Light Gradient Boosting Machine,23.7935,980.8767,29.5674,-0.1419,0.1699,0.1450,0.2193
ada,AdaBoost Regressor,24.0594,962.3062,29.1163,-0.1847,0.1676,0.1492,0.0727
gbr,Gradient Boosting Regressor,25.0554,1060.4871,29.8569,-0.1870,0.1695,0.1535,0.0660
rf,Random Forest Regressor,24.6026,996.6377,29.0843,-0.2160,0.1671,0.1498,0.1700
xgboost,Extreme Gradient Boosting,26.0463,1144.2354,31.0708,-0.2891,0.1745,0.1596,0.4060
omp,Orthogonal Matching Pursuit,25.3471,1123.1213,32.1988,-0.3398,0.1854,0.1576,0.0147
br,Bayesian Ridge,25.3472,1130.8385,32.2756,-0.3437,0.1859,0.1583,0.0167
dummy,Dummy Regressor,25.3531,1137.5242,32.2790,-0.3490,0.1859,0.1591,0.0147


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extra Trees Regressor,22.8615,912.3257,30.2047,0.3430,0.1733,0.1378


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,15.6374,409.6167,20.2390,-1.1499,0.0975,0.0721
1,23.0431,622.9453,24.9589,-0.1102,0.1338,0.1268
2,23.9385,707.2272,26.5937,0.2080,0.1483,0.1350
3,19.5991,692.8920,26.3228,-0.1311,0.1258,0.0882
4,22.5660,825.5306,28.7320,-0.4781,0.1802,0.1571
5,36.0519,2777.0940,52.6981,0.0904,0.4947,0.4674
6,27.9991,1133.5510,33.6682,0.0398,0.1671,0.1420
7,22.3069,609.0773,24.6795,-0.4961,0.1331,0.1241
8,24.6328,1610.8283,40.1351,0.1424,0.2017,0.1308


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extra Trees Regressor,22.8615,912.3257,30.2047,0.3430,0.1733,0.1378


¡Proceso terminado!
ExtraTreesRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región de Atacama


El ejercicio de modelado para la Región de Atacama presenta resultados desafiantes, seleccionando al algoritmo Extra Trees Regressor como la mejor opción disponible, aunque con un desempeño limitado. Si bien el modelo optimizado alcanza un $R^2$ de 0.34 en el conjunto de prueba final y mantiene un MAPE promedio del 14.33% (aceptable para estimaciones generales), la validación cruzada revela serios problemas estructurales. El $R^2$ promedio durante el entrenamiento fue negativo (-0.10), indicando que en la mayoría de los segmentos el modelo fue incapaz de superar a una simple media aritmética. Además, se observa un quiebre severo en el Fold 5 (MAPE 46%), lo que confirma la presencia de valores atípicos o cambios de tendencia abruptos que este modelo no logra capturar. En conclusión, aunque es la "mejor" herramienta disponible actualmente, su fiabilidad es baja y se recomienda su uso solo como referencia indicativa, no definitiva.

In [18]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num", "Región de Coquimbo"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Coquimbo", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Coquimbo")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Coquimbo
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
en,Elastic Net,46.5077,4556.1680,63.2924,0.1168,0.2161,0.1697,0.0160
lasso,Lasso Regression,46.5122,4558.1293,63.3050,0.1163,0.2162,0.1697,0.0160
llar,Lasso Least Angle Regression,46.5122,4558.1293,63.3050,0.1163,0.2162,0.1697,0.0140
lr,Linear Regression,46.5083,4558.1040,63.3023,0.1161,0.2162,0.1697,0.0160
ridge,Ridge Regression,46.5083,4558.0556,63.3020,0.1161,0.2162,0.1697,0.0153
lar,Least Angle Regression,46.5083,4558.1040,63.3023,0.1161,0.2162,0.1697,0.0147
br,Bayesian Ridge,46.8578,4594.6362,63.6983,0.1080,0.2165,0.1709,0.0147
huber,Huber Regressor,45.7665,4520.3012,62.9853,0.0999,0.2146,0.1623,0.0213
omp,Orthogonal Matching Pursuit,46.9535,4612.2755,63.8788,0.0968,0.2167,0.1710,0.0147
lightgbm,Light Gradient Boosting Machine,47.9068,4359.7218,62.9236,0.0924,0.2159,0.1739,0.1953


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Elastic Net,62.5548,9795.7753,98.9736,0.1858,0.2932,0.2165


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,45.2090,5794.3286,76.1205,-0.0569,0.2352,0.1321
1,45.4302,3269.4768,57.1793,0.4233,0.1551,0.1235
2,37.2275,1874.9453,43.3006,0.4275,0.1358,0.1116
3,53.2862,8291.7705,91.0592,-0.0990,0.2644,0.1514
4,37.0993,2327.3647,48.2428,0.5092,0.1701,0.1259
5,63.3816,9567.6301,97.8143,-0.5424,0.3935,0.3461
6,53.9557,4882.2315,69.8730,0.4422,0.2120,0.1872
7,28.9208,1004.7370,31.6976,0.4261,0.1253,0.1183
8,42.1030,2998.3544,54.7572,-0.0601,0.2555,0.2219


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Elastic Net,62.4768,9781.0356,98.8991,0.1871,0.2925,0.2158


¡Proceso terminado!
ElasticNet(random_state=123)
Proceso finalizado para Región de Coquimbo


El estudio predictivo para la Región de Coquimbo arrojó un resultado distintivo: la parsimonia prevaleció sobre la complejidad, seleccionando a Elastic Net (regresión lineal regularizada) como el modelo óptimo, mientras que algoritmos no lineales como Random Forest o XGBoost colapsaron debido al sobreajuste. No obstante, el ajuste estadístico es precario; con un $R^2$ promedio de solo 0.12, el modelo apenas logra explicar la varianza de los datos, funcionando marginalmente mejor que una media simple. Aunque el MAPE del 16.98% sugiere un error operativo manejable, la validación cruzada expone una inestabilidad severa: el modelo oscila entre ajustes aceptables (Fold 14, $R^2$ 0.62) y fallos catastróficos (Fold 9, $R^2$ -1.56). Esto indica que la dinámica de Coquimbo no sigue una tendencia lineal clara ni patrones estacionales robustos detectables con las variables actuales, sugiriendo que la serie depende fuertemente de factores exógenos no modelados.

In [19]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num", "Región de Valparaíso"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Valparaíso", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Valparaíso")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Valparaíso
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,148.4584,43605.0028,195.4735,0.4370,0.1382,0.1086,0.1713
gbr,Gradient Boosting Regressor,149.0942,44390.6478,198.9124,0.4146,0.1420,0.1091,0.0647
et,Extra Trees Regressor,152.6079,46366.0932,200.2460,0.3925,0.1429,0.1124,0.1267
xgboost,Extreme Gradient Boosting,162.3038,48593.1290,206.3042,0.3557,0.1451,0.1181,0.4033
lightgbm,Light Gradient Boosting Machine,177.2593,59453.4060,228.3259,0.2608,0.1619,0.1311,0.2113
ada,AdaBoost Regressor,181.5236,57000.9904,229.1299,0.2492,0.1656,0.1377,0.0867
knn,K Neighbors Regressor,195.5019,67353.2906,247.7197,0.1434,0.1772,0.1406,0.0407
dt,Decision Tree Regressor,189.3540,66128.0810,242.3653,0.1412,0.1676,0.1368,0.0153
omp,Orthogonal Matching Pursuit,219.9488,85531.0803,285.4427,-0.1100,0.2058,0.1673,0.0133
br,Bayesian Ridge,220.2639,85858.3218,286.0393,-0.1111,0.2063,0.1676,0.0140


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,140.8127,31776.5394,178.2598,0.7505,0.1212,0.0991


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,112.3757,21936.7839,148.1107,0.7125,0.1328,0.1025
1,241.7042,79760.7815,282.4195,0.5162,0.1953,0.1684
2,214.4021,96784.9313,311.1028,-0.1527,0.1790,0.1227
3,118.7679,21363.7078,146.1633,0.3946,0.1130,0.0998
4,152.1058,40626.7963,201.5609,0.3095,0.1276,0.0896
5,191.4320,133279.5112,365.0747,-0.6108,0.3462,0.2534
6,140.1658,22688.9645,150.6286,0.4851,0.1224,0.1174
7,167.7541,48510.5051,220.2510,0.5566,0.1573,0.1232
8,142.1252,29951.5464,173.0652,0.3610,0.1313,0.1090


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,140.8127,31776.5394,178.2598,0.7505,0.1212,0.0991


¡Proceso terminado!
RandomForestRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región de Valparaíso


El modelado para la Región de Valparaíso muestra un desempeño prometedor pero volátil, seleccionando al Random Forest Regressor como el estimador dominante frente a la nula capacidad explicativa de los modelos lineales. El algoritmo exhibe una dicotomía interesante: aunque su promedio en validación cruzada es moderado ($R^2$ 0.37), su rendimiento en el conjunto de prueba final fue excelente, alcanzando un $R^2$ de 0.75 y un MAPE del 9.91%, lo que lo hace altamente operativo para proyecciones actuales. Sin embargo, la estabilidad histórica es preocupante; el modelo oscila entre la precisión casi perfecta (Fold 10, $R^2$ 0.81) y el colapso total en ventanas específicas (Fold 5, $R^2$ -0.61). Esto indica que, si bien el modelo entiende bien la dinámica reciente de la región, existen anomalías o quiebres estructurales en el pasado que dificultan un aprendizaje homogéneo en toda la serie temporal.

In [20]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num", "Región Metropolitana"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región Metropolitana", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región Metropolitana")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región Metropolitana
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,274.2638,149555.0277,355.1892,0.7561,0.1146,0.0915,0.0673
rf,Random Forest Regressor,283.9579,170111.4343,366.2096,0.7355,0.1164,0.0951,0.1720
et,Extra Trees Regressor,301.6228,171904.9934,377.5484,0.7252,0.1189,0.0992,0.1273
xgboost,Extreme Gradient Boosting,302.1314,167951.1733,380.8141,0.7189,0.1219,0.0999,0.4007
ada,AdaBoost Regressor,309.8146,208269.2876,412.1598,0.6535,0.1320,0.1065,0.0493
lightgbm,Light Gradient Boosting Machine,333.9306,231456.1296,428.6438,0.6462,0.1402,0.1165,0.2127
knn,K Neighbors Regressor,363.5797,238743.4070,453.4555,0.5896,0.1465,0.1199,0.0393
dt,Decision Tree Regressor,400.6206,258279.5476,481.5490,0.5606,0.1484,0.1260,0.0140
omp,Orthogonal Matching Pursuit,442.1150,353821.2222,558.1154,0.4146,0.1824,0.1566,0.0133
br,Bayesian Ridge,444.9853,358321.3520,561.7854,0.4099,0.1839,0.1577,0.0140


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,212.9740,73292.8636,270.7265,0.9407,0.1012,0.0753


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,163.3407,35276.9130,187.8215,0.9575,0.0651,0.0543
1,387.3001,188374.7633,434.0216,0.8200,0.1205,0.1101
2,332.6906,183297.0399,428.1320,0.7684,0.1351,0.0961
3,228.2189,74425.2096,272.8098,0.5174,0.0783,0.0684
4,227.6550,93042.3602,305.0285,0.8719,0.0867,0.0673
5,435.5026,727005.0402,852.6459,0.2490,0.3921,0.2936
6,172.5778,40194.1723,200.4848,0.9363,0.0704,0.0645
7,167.6646,62578.3914,250.1567,0.9103,0.0956,0.0607
8,254.5361,84361.5847,290.4507,0.8201,0.0965,0.0884


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,268.8536,121482.8210,348.5439,0.9017,0.1141,0.0881


¡Proceso terminado!
GradientBoostingRegressor(random_state=123)
Proceso finalizado para Región Metropolitana


El modelado para la Región Metropolitana presenta el desempeño más robusto de todo el estudio, seleccionando al Gradient Boosting Regressor como el algoritmo óptimo con una capacidad predictiva excelente. El modelo alcanzó un $R^2$ de 0.90 en el conjunto de prueba final, lo que indica que logra explicar el 90% de la variabilidad de los siniestros, acompañado de un MAPE del 8.81%, un nivel de precisión muy alto para fines operativos. A diferencia de las regiones del norte, aquí la estacionalidad y tendencia son claras, permitiendo que el modelo supere ampliamente a los enfoques lineales. Sin embargo, se confirma un patrón sistémico en los datos: el Fold 5 vuelve a presentar una caída drástica de rendimiento ($R^2$ 0.24 frente a promedios de >0.80), lo que corrobora la existencia de un evento extremo o anómalo en un periodo específico de la historia que desafía la capacidad de generalización de cualquier algoritmo entrenado.

In [21]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región del Libertador Bernardo O'higgins"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región del Libertador Bernardo O'higgins", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región del Libertador Bernardo O'higgins")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región del Libertador Bernardo O'higgins
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,190.1406,81162.9154,254.7143,0.2648,0.1965,0.1577,0.1880
gbr,Gradient Boosting Regressor,201.9692,90677.5412,267.3239,0.2079,0.2026,0.1661,0.0560
et,Extra Trees Regressor,198.0014,82696.0675,260.6524,0.1468,0.2042,0.1671,0.1493
knn,K Neighbors Regressor,218.3819,96621.2157,279.8624,0.0962,0.2150,0.1736,0.0400
lightgbm,Light Gradient Boosting Machine,211.5497,91908.8092,273.1135,0.0686,0.2135,0.1804,0.1873
ada,AdaBoost Regressor,230.7744,100806.8139,294.6181,-0.0785,0.2270,0.1946,0.0693
xgboost,Extreme Gradient Boosting,226.8935,104562.0455,294.4484,-0.0932,0.2239,0.1862,0.4120
huber,Huber Regressor,251.5388,109185.6793,309.3422,-0.1924,0.2440,0.2118,0.0220
en,Elastic Net,250.8908,109404.7927,310.4482,-0.2468,0.2456,0.2156,0.0140
llar,Lasso Least Angle Regression,250.8735,109491.9005,310.5353,-0.2485,0.2457,0.2155,0.0167


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,196.6946,57514.1433,239.8211,0.5424,0.1978,0.1695


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,308.7506,309548.2346,556.3706,0.2800,0.3061,0.1838
1,208.5362,55579.6827,235.7534,0.7413,0.2068,0.1793
2,205.4674,74634.2623,273.1927,0.4161,0.1741,0.1384
3,138.0116,27232.5123,165.0228,-0.7446,0.1422,0.1267
4,110.8240,14480.0425,120.3330,0.5490,0.0874,0.0802
5,247.2757,130730.0247,361.5661,0.0565,0.4118,0.3577
6,132.9445,26741.0606,163.5269,0.4765,0.1532,0.1375
7,195.0295,53565.8406,231.4430,0.4936,0.2387,0.1964
8,151.0639,24554.5786,156.6990,-0.1522,0.1516,0.1534


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,200.3512,55894.5069,236.4202,0.5553,0.1918,0.1730


¡Proceso terminado!
RandomForestRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región del Libertador Bernardo O'higgins


El modelado para la Región del Libertador Bernardo O'Higgins presenta un escenario de moderada incertidumbre. Se seleccionó al Random Forest Regressor como el modelo dominante, dado que los algoritmos lineales fueron incapaces de capturar la dinámica de los datos ($R^2$ negativo). Aunque el modelo muestra un ajuste razonable en el conjunto de prueba final ($R^2$ 0.56) y un MAPE promedio del 16.20%, la validación histórica es preocupante. La estabilidad es baja, evidenciada por caídas drásticas en el rendimiento durante el Fold 3 ($R^2$ -0.74) y nuevamente en el Fold 5, lo que sugiere que la región experimentó eventos atípicos en el pasado que rompen la tendencia aprendida, limitando la fiabilidad del modelo para escenarios extremos.

In [22]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región del Maule"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región del Maule", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región del Maule")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región del Maule
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,111.4636,24383.5401,144.5693,0.5033,0.1838,0.1486,0.1847
gbr,Gradient Boosting Regressor,117.5875,27123.5115,153.6243,0.4812,0.1953,0.1570,0.0593
xgboost,Extreme Gradient Boosting,116.4271,27409.3616,153.0818,0.4720,0.1967,0.1552,0.5073
et,Extra Trees Regressor,125.0631,28987.8361,161.5084,0.3804,0.1993,0.1631,0.1420
lightgbm,Light Gradient Boosting Machine,133.7182,31789.4732,167.2582,0.3066,0.2077,0.1763,0.1887
ada,AdaBoost Regressor,136.8191,33807.9383,175.9929,0.2675,0.2190,0.1845,0.0593
dt,Decision Tree Regressor,140.0175,39118.2556,187.3874,0.0667,0.2290,0.1813,0.0147
knn,K Neighbors Regressor,186.2146,58896.2672,228.7322,-0.0521,0.2731,0.2234,0.0393
omp,Orthogonal Matching Pursuit,220.8065,70390.0201,254.9561,-0.4113,0.3076,0.2906,0.0140
br,Bayesian Ridge,188.8876,54974.7100,226.3038,-0.4633,0.2828,0.2505,0.0133


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,158.7273,68424.5105,261.5808,0.5121,0.2334,0.1711


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,167.7764,61321.3370,247.6315,0.2944,0.2827,0.2068
1,141.3526,30357.4464,174.2339,0.7434,0.2233,0.1830
2,159.0350,33944.2296,184.2396,0.1877,0.2120,0.1762
3,110.2342,22372.7525,149.5752,-0.1887,0.1771,0.1416
4,80.8279,12231.8060,110.5975,0.6913,0.1131,0.0830
5,162.0048,42260.4787,205.5735,0.5232,0.4028,0.3595
6,106.8793,14284.7053,119.5186,0.3670,0.1736,0.1679
7,97.5201,17926.3873,133.8895,0.2811,0.1658,0.1217
8,48.5837,3621.7979,60.1814,0.0703,0.0865,0.0718


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,158.7273,68424.5105,261.5808,0.5121,0.2334,0.1711


¡Proceso terminado!
RandomForestRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región del Maule


El modelado para la Región del Maule confirma la superioridad de los métodos de ensamble, posicionando al Random Forest Regressor como el algoritmo óptimo frente al colapso total de los modelos lineales. Aunque el desempeño general es moderado, con un $R^2$ de 0.51 en el conjunto de prueba y un MAPE del 17.11%, el análisis de validación cruzada revela una dicotomía extrema en la calidad de los datos. Coexisten periodos de predictibilidad casi perfecta, como los Folds 10 y 11 (con $R^2$ superiores a 0.91), frente a "zonas muertas" como el Fold 3 ($R^2$ -0.19), donde el modelo pierde toda capacidad de rastreo. Esta volatilidad sugiere que, si bien el patrón estacional es muy fuerte y fácil de modelar en años normales, existen años específicos con anomalías severas (posiblemente megasequías o incendios forestales masivos) que rompen la tendencia histórica y requieren variables exógenas para ser anticipados.

In [23]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región del Ñuble"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región del Ñuble", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región del Ñuble")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región del Ñuble
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,82.8062,14448.5621,103.6296,0.3911,0.1916,0.1676,0.1873
lightgbm,Light Gradient Boosting Machine,88.8936,16605.2471,113.1532,0.3725,0.2080,0.1819,0.2320
xgboost,Extreme Gradient Boosting,90.7536,16336.1773,112.7406,0.3696,0.2048,0.1788,0.4167
et,Extra Trees Regressor,83.5120,14851.3385,106.1690,0.3486,0.1938,0.1672,0.1393
dt,Decision Tree Regressor,100.1492,18887.4444,126.0095,0.2055,0.2270,0.1954,0.0133
huber,Huber Regressor,103.8768,20948.9938,130.5154,0.1772,0.2475,0.2156,0.0200
ada,AdaBoost Regressor,100.4846,19701.0053,124.6270,0.1717,0.2303,0.2127,0.0607
knn,K Neighbors Regressor,108.1400,22317.0156,136.1600,0.1242,0.2515,0.2124,0.0400
br,Bayesian Ridge,105.2446,20874.7125,130.5604,0.1219,0.2511,0.2254,0.0127
en,Elastic Net,105.1837,20834.4600,130.5554,0.1187,0.2515,0.2253,0.0133


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,84.6692,20280.5438,142.4098,0.5552,0.2368,0.1627


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,70.0445,6093.7913,78.0627,0.5918,0.1907,0.1628
1,98.4269,20978.0484,144.8380,0.6791,0.2493,0.1769
2,79.7626,9809.8336,99.0446,0.6461,0.1780,0.1490
3,69.1126,7758.3785,88.0817,0.5360,0.1460,0.1152
4,67.7258,5922.8459,76.9600,0.6456,0.1261,0.1139
5,127.7538,37006.2846,192.3702,-0.1517,0.5217,0.4901
6,80.3077,7762.8133,88.1068,0.4339,0.2066,0.2023
7,31.5956,1293.3614,35.9633,0.9386,0.0755,0.0682
8,35.1879,1713.8102,41.3982,0.5070,0.1031,0.0920


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,87.0342,19419.4221,139.3536,0.5740,0.2511,0.1809


¡Proceso terminado!
RandomForestRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región del Ñuble


El modelado predictivo para la Región del Ñuble confirma al Random Forest Regressor como la herramienta más eficiente, superando la inestabilidad de los algoritmos lineales y de boosting. Aunque el rendimiento general es moderado, con un $R^2$ de 0.57 y un MAPE del 18.09% en el conjunto de prueba, el modelo demuestra ser operativamente valioso en condiciones normales. Destaca la existencia de periodos de alta predictibilidad, como el Fold 7 ($R^2$ 0.93) y Fold 9 ($R^2$ 0.86), donde el modelo captura la dinámica casi a la perfección. Sin embargo, la fiabilidad se ve comprometida por "puntos ciegos" recurrentes, específicamente en el Fold 5 y el Fold 14, donde el ajuste colapsa. Esto refuerza la hipótesis de que la región sufre perturbaciones esporádicas (probablemente incendios forestales extremos fuera de temporada o cambios metodológicos) que rompen la estructura histórica de los datos.

In [24]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región del Bío Bío"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región del Bío Bío", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región del Bío Bío")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región del Bío Bío
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,158.7608,52683.6077,206.9791,0.4065,0.1533,0.1222,0.0700
et,Extra Trees Regressor,161.8468,51888.8341,210.7434,0.3447,0.1603,0.1293,0.1553
lightgbm,Light Gradient Boosting Machine,165.2926,55603.3285,214.9300,0.3413,0.1651,0.1328,0.2000
rf,Random Forest Regressor,159.1481,52236.9031,207.9705,0.3214,0.1568,0.1265,0.1793
xgboost,Extreme Gradient Boosting,175.3069,61723.3643,224.8854,0.2299,0.1688,0.1383,0.4507
ada,AdaBoost Regressor,189.7276,67870.8384,239.1457,0.1157,0.1806,0.1535,0.0807
huber,Huber Regressor,196.9033,73611.8259,253.5277,-0.0075,0.2008,0.1580,0.0213
br,Bayesian Ridge,199.8902,73575.6102,255.0170,-0.0176,0.2008,0.1633,0.0147
en,Elastic Net,199.3609,73399.0767,254.7151,-0.0177,0.2007,0.1630,0.0160
llar,Lasso Least Angle Regression,198.4775,73392.4334,254.8115,-0.0283,0.2014,0.1623,0.0167


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,153.4561,57799.2846,240.4148,0.5252,0.1708,0.1200


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,93.3092,13794.8949,117.4517,0.4490,0.1153,0.0896
1,273.6016,114654.6336,338.6069,0.3883,0.2221,0.1766
2,135.6682,22484.0745,149.9469,0.3511,0.1265,0.1184
3,134.1608,30127.6881,173.5733,0.3125,0.1246,0.0979
4,95.8554,20450.0610,143.0037,0.6444,0.0926,0.0651
5,147.5599,72555.3287,269.3610,0.1939,0.3056,0.2106
6,149.3661,31449.2565,177.3394,0.2050,0.1792,0.1551
7,109.3980,18020.4204,134.2402,0.5568,0.1293,0.1090
8,134.7638,37759.4881,194.3180,-0.8504,0.1878,0.1473


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,153.4561,57799.2846,240.4148,0.5252,0.1708,0.1200


¡Proceso terminado!
GradientBoostingRegressor(random_state=123)
Proceso finalizado para Región del Bío Bío


El modelado para la Región del Bío Bío seleccionó al Gradient Boosting Regressor como el estimador más equilibrado, logrando un $R^2$ de 0.53 en el conjunto de prueba y un MAPE del 12.00%. Aunque el error porcentual es bajo, lo que hace al modelo operativo, la validación cruzada expone una volatilidad extrema: el modelo es capaz de explicar el 81% de la varianza en periodos estables (Fold 11), pero falla catastróficamente en otros (Fold 8 con $R^2$ -0.85 y Fold 13 con errores masivos). Esto sugiere que la Región del Bío Bío, siendo una zona de alta actividad forestal, sufre de "mega-eventos" (incendios masivos fuera de escala) que rompen cualquier tendencia aprendida. Adicionalmente, el Fold 5 vuelve a mostrar un desempeño deficiente ($R^2$ 0.19), sumándose al patrón anómalo detectado en las otras regiones.

In [25]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región de la Araucanía"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de la Araucanía", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de la Araucanía")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de la Araucanía
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,170.6745,63395.7805,228.7098,0.5398,0.1745,0.1423,0.0607
xgboost,Extreme Gradient Boosting,172.5728,63893.2059,227.3056,0.5396,0.1740,0.1440,0.4573
lightgbm,Light Gradient Boosting Machine,180.4443,69617.0775,243.7930,0.5251,0.2008,0.1591,0.1847
et,Extra Trees Regressor,167.0081,55611.6945,218.8593,0.4819,0.1718,0.1430,0.1373
rf,Random Forest Regressor,178.4966,66306.3299,237.6198,0.4357,0.1857,0.1493,0.1853
ada,AdaBoost Regressor,209.3902,77166.8374,261.1198,0.2019,0.2143,0.1894,0.0633
dt,Decision Tree Regressor,238.1429,122711.9937,319.5467,0.0001,0.2344,0.1916,0.0147
knn,K Neighbors Regressor,241.9317,129480.4265,327.1970,-0.1489,0.2507,0.1914,0.0380
huber,Huber Regressor,239.7556,114827.0652,315.3933,-0.3544,0.2546,0.2047,0.0213
omp,Orthogonal Matching Pursuit,306.1570,163664.9535,376.9354,-0.5259,0.3097,0.2734,0.0133


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,121.7819,30834.8012,175.5984,0.7921,0.1492,0.1061


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,83.0904,12207.1991,110.4862,0.6301,0.1435,0.1032
1,242.4621,99538.8026,315.4977,0.6111,0.2464,0.1855
2,137.3040,27701.0349,166.4363,0.7393,0.1776,0.1408
3,211.3032,54602.7472,233.6723,0.0641,0.2085,0.1852
4,231.5533,115367.3948,339.6578,0.2120,0.2652,0.1681
5,244.5875,121210.3816,348.1528,0.1048,0.3816,0.3231
6,171.3490,61331.1233,247.6512,0.6199,0.2110,0.1639
7,147.0533,31019.4020,176.1233,0.4172,0.2468,0.1762
8,54.1899,4735.7219,68.8166,0.4134,0.0789,0.0618


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,121.7819,30834.8012,175.5984,0.7921,0.1492,0.1061


¡Proceso terminado!
GradientBoostingRegressor(random_state=123)
Proceso finalizado para Región de la Araucanía


El modelado para la Región de la Araucanía presenta resultados mixtos, seleccionando al Gradient Boosting Regressor como el modelo más apto. El desempeño en el conjunto de prueba final es sorprendentemente bueno, con un $R^2$ de 0.79 y un MAPE del 10.61%, lo que sugiere que el modelo aprendió muy bien la dinámica reciente. Sin embargo, la historia es diferente en la validación cruzada: el promedio de $R^2$ baja a 0.49, mostrando una inestabilidad significativa. Esto se debe a "zonas de falla" muy marcadas, como el Fold 3 ($R^2$ 0.06) y nuevamente el Fold 5 ($R^2$ 0.10), donde el modelo pierde su capacidad predictiva. Esto refuerza la teoría de que existen anomalías históricas (como el "conflicto del Fold 5") que, si se limpian, permitirían que este modelo sea altamente preciso y confiable.

In [26]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región de los Ríos"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de los Ríos", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de los Ríos")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de los Ríos
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,55.5885,5395.9497,67.5174,0.6100,0.1758,0.1515,0.1840
gbr,Gradient Boosting Regressor,57.1499,5651.6478,69.9672,0.5465,0.1814,0.1537,0.0580
dt,Decision Tree Regressor,64.6778,6840.3222,77.9046,0.5344,0.1944,0.1671,0.0140
xgboost,Extreme Gradient Boosting,62.7052,6364.2861,74.4334,0.5210,0.1929,0.1685,0.5660
lightgbm,Light Gradient Boosting Machine,63.9901,7797.2019,80.3572,0.5015,0.2008,0.1736,0.1867
et,Extra Trees Regressor,57.2256,6017.2227,72.7676,0.4876,0.1862,0.1571,0.1387
knn,K Neighbors Regressor,71.1962,9501.2397,89.6080,0.3516,0.2316,0.1892,0.0400
huber,Huber Regressor,76.3399,11226.2613,94.5748,0.3450,0.2453,0.2080,0.0220
en,Elastic Net,77.8247,11129.6861,95.0278,0.3390,0.2461,0.2169,0.0147
ridge,Ridge Regression,77.9288,11141.9555,95.1379,0.3377,0.2465,0.2170,0.0140


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,38.6542,2538.1511,50.3801,0.8211,0.1358,0.1041


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,54.3066,4066.6062,63.7699,0.4930,0.1773,0.1508
1,58.0105,6314.0482,79.4610,0.7704,0.1664,0.1377
2,46.8399,2401.0115,49.0001,0.8484,0.1487,0.1327
3,30.1016,2143.5297,46.2983,0.8501,0.0903,0.0626
4,32.6215,1410.8019,37.5606,0.8489,0.0937,0.0814
5,96.2496,14983.3536,122.4065,0.3533,0.4146,0.3769
6,66.3214,7575.8013,87.0391,0.3585,0.2600,0.2358
7,40.0580,2033.5584,45.0950,0.8098,0.1287,0.1219
8,45.5806,4410.6174,66.4125,0.1849,0.2283,0.1826


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,38.6542,2538.1511,50.3801,0.8211,0.1358,0.1041


¡Proceso terminado!
RandomForestRegressor(n_jobs=-1, random_state=123)
Proceso finalizado para Región de los Ríos


El modelado para la Región de Los Ríos muestra un comportamiento muy interesante y prometedor, seleccionando al Random Forest Regressor como el modelo más competente. En el conjunto de prueba final, el modelo alcanzó un excelente $R^2$ de 0.82 con un MAPE del 10.41%, lo que indica una alta precisión operativa. Sin embargo, la validación cruzada cuenta una historia de "doble cara": existen periodos de predictibilidad casi absoluta, como el Fold 12 con un impresionante $R^2$ de 0.97 (prácticamente perfecto), frente a caídas bruscas de rendimiento, nuevamente en el Fold 5 ($R^2$ 0.35) y Fold 8 ($R^2$ 0.18). Esto sugiere que la región tiene una dinámica muy limpia y estacional la mayor parte del tiempo, pero sufre de perturbaciones puntuales que desajustan el modelo temporalmente.

In [27]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región de los Lagos"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de los Lagos", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de los Lagos")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de los Lagos
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,107.5351,24162.1020,143.2475,-0.0016,0.1413,0.1079,0.2107
ada,AdaBoost Regressor,109.1335,22542.9400,141.9866,-0.0418,0.1367,0.1095,0.0753
gbr,Gradient Boosting Regressor,109.5520,20789.2586,136.2759,-0.0421,0.1296,0.1069,0.0620
et,Extra Trees Regressor,111.9256,22288.9580,142.2862,-0.0645,0.1347,0.1092,0.1387
rf,Random Forest Regressor,107.5788,21643.5548,136.8930,-0.0751,0.1307,0.1062,0.1847
huber,Huber Regressor,115.7948,26119.4805,153.2282,-0.1452,0.1497,0.1161,0.0207
en,Elastic Net,115.3990,26398.8534,153.9900,-0.1677,0.1508,0.1166,0.0133
dummy,Dummy Regressor,120.4349,28088.4668,159.4654,-0.1707,0.1559,0.1219,0.0127
omp,Orthogonal Matching Pursuit,122.5288,28880.8957,160.6422,-0.1729,0.1570,0.1243,0.0127
llar,Lasso Least Angle Regression,115.5090,26426.1225,154.0876,-0.1759,0.1510,0.1166,0.0133


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,109.0730,20859.9680,144.4298,0.3042,0.1402,0.1080


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,158.3989,57063.4274,238.8795,-0.1352,0.1943,0.1242
1,135.7098,29521.6838,171.8188,0.1242,0.1516,0.1200
2,123.4158,22666.9062,150.5553,0.0684,0.1573,0.1343
3,75.9879,10544.6935,102.6874,-0.3712,0.0907,0.0650
4,88.8441,11777.0109,108.5219,0.1756,0.0980,0.0800
5,147.5128,52872.3945,229.9400,0.0440,0.3006,0.2250
6,132.4603,25557.9317,159.8685,-0.0406,0.1584,0.1344
7,85.1420,14647.6675,121.0275,0.1053,0.1357,0.0991
8,77.2668,13365.0799,115.6074,-0.4932,0.1266,0.0904


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,119.8547,23395.0471,152.9544,0.2196,0.1530,0.1217


¡Proceso terminado!
LGBMRegressor(device='gpu', n_jobs=-1, random_state=123)
Proceso finalizado para Región de los Lagos


El modelado para la Región de los Lagos arroja los resultados más débiles de todo el estudio hasta el momento. Aunque se seleccionó LightGBM como el mejor estimador, su capacidad explicativa es prácticamente nula: el $R^2$ promedio en validación cruzada es 0.01, lo que significa que el modelo no logra superar el rendimiento de simplemente predecir el promedio histórico. Aunque el MAPE del 12.17% parece aceptable a primera vista, es un "falso positivo" operativo; el modelo no está anticipando subidas ni bajadas, solo se mantiene en una línea base conservadora. La inestabilidad es sistémica: fallos graves en los Folds 0, 3, 8 y 13 (con $R^2$ negativos) sugieren que la dinámica de incendios en esta región no obedece a las mismas variables temporales que el resto del país, o que el ruido estadístico supera a la señal.

In [28]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región de Aysén del General Carlos Ibáñez del Campo"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Aysén del General Carlos Ibáñez del Campo", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Aysén del General Carlos Ibáñez del Campo")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Aysén del General Carlos Ibáñez del Campo
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
ada,AdaBoost Regressor,15.6652,352.9030,18.2834,0.0307,0.2297,0.2138,0.0387
rf,Random Forest Regressor,15.2718,360.3350,18.4064,0.0304,0.2332,0.2078,0.2040
gbr,Gradient Boosting Regressor,15.8518,381.7069,18.9130,-0.0166,0.2335,0.2082,0.0653
lightgbm,Light Gradient Boosting Machine,15.7974,435.2117,19.8894,-0.0664,0.2631,0.2428,0.1947
et,Extra Trees Regressor,15.4769,386.6182,19.0777,-0.0738,0.2434,0.2163,0.1580
dummy,Dummy Regressor,17.1574,470.6214,20.8307,-0.1457,0.2756,0.2649,0.0167
br,Bayesian Ridge,17.2902,480.2068,21.0005,-0.1626,0.2775,0.2686,0.0173
omp,Orthogonal Matching Pursuit,17.2076,482.2289,21.0367,-0.1758,0.2780,0.2689,0.0153
knn,K Neighbors Regressor,16.5565,426.7028,19.8972,-0.1979,0.2652,0.2496,0.0400
llar,Lasso Least Angle Regression,17.3774,487.6348,21.2171,-0.2054,0.2802,0.2710,0.0147


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,AdaBoost Regressor,16.3097,448.8663,21.1865,0.3330,0.2824,0.2265


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,16.3404,309.2891,17.5866,0.3310,0.1846,0.1758
1,14.5768,341.3305,18.4751,0.1580,0.2238,0.1834
2,16.1408,331.5972,18.2098,-0.0611,0.2158,0.2130
3,16.0696,340.4523,18.4514,0.1077,0.2281,0.2034
4,17.8045,408.9835,20.2233,0.2986,0.2505,0.2342
5,22.2440,829.7792,28.8059,0.0043,0.5029,0.5330
6,15.3145,289.9803,17.0288,-0.0833,0.1994,0.1889
7,12.5184,219.6053,14.8191,0.5077,0.1773,0.1520
8,17.8128,346.3041,18.6092,0.5156,0.3602,0.3738


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,AdaBoost Regressor,15.4343,425.9489,20.6385,0.3671,0.2526,0.2060


¡Proceso terminado!
AdaBoostRegressor(random_state=123)
Proceso finalizado para Región de Aysén del General Carlos Ibáñez del Campo


El modelado para la Región de Aysén arroja resultados críticos. Se seleccionó AdaBoost Regressor como el "mejor" modelo, pero el rendimiento es estadísticamente insuficiente. El $R^2$ promedio de validación cruzada es de solo 0.12, lo que significa que el modelo apenas supera al azar. Aunque el error absoluto (MAE) es bajo (14.95), esto se debe a que la magnitud de incendios en Aysén es pequeña, no a que el modelo sea preciso. El Fold 5 vuelve a aparecer con un error porcentual masivo (MAPE 53.3%) y el Fold 14 colapsa ($R^2$ -0.79), confirmando que el modelo no logra generalizar la estacionalidad.

In [29]:
from pycaret.regression import *


# 2. CORRECCIÓN CRÍTICA DE DATOS:
# Agrupamos por Año Y Mes. Si solo usas "anho", conviertes todo el año en un solo dato.
df_totales = df.groupby(["anho", "mes_num"]).sum(numeric_only=True).reset_index() 

# Selección de columnas (Asegúrate de usar "anho" si así se llama en tu df original)
data_py = df_totales[["anho", "mes_num","Región de Magallanes y de la Antártica Chilena"]].copy()


# 2. Ejecutamos PyCaret dentro del bloque silencioso
print("Entrenando modelos, por favor espera... (mensajes ocultos)")

with suppress_output():
    # Agregamos verbose=False como buena práctica extra
    reg = setup(
        data=data_py, 
        target="Región de Magallanes y de la Antártica Chilena", 
        session_id=123,
        train_size=0.8,
        fold_shuffle=False,
        use_gpu=True,
        fold=a,
    )
    
    # LightGBM suele ser ruidoso, verbose=False ayuda, pero el bloque 'with' es el que realmente lo calla
    best_model = compare_models()
    
    predictions = predict_model(best_model)
    
    tuned_model = tune_model(best_model, optimize='R2')
    
    predictions_tuned = predict_model(tuned_model)

print("¡Proceso terminado!")
print(best_model)

print("Proceso finalizado para Región de Magallanes y de la Antártica Chilena")

Entrenando modelos, por favor espera... (mensajes ocultos)


,Description,Value
0,Session id,123
1,Target,Región de Magallanes y de la Antártica Chilena
2,Target type,Regression
3,Original data shape,"(129, 3)"
4,Transformed data shape,"(129, 3)"
5,Transformed train set shape,"(103, 3)"
6,Transformed test set shape,"(26, 3)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,17.0114,493.0381,21.1163,-0.4216,0.2044,0.1734,0.2020
br,Bayesian Ridge,17.0908,520.0476,21.6853,-0.4739,0.2154,0.1802,0.0193
omp,Orthogonal Matching Pursuit,16.9638,517.1390,21.6019,-0.4774,0.2143,0.1787,0.0187
huber,Huber Regressor,16.7839,532.7251,21.8338,-0.4942,0.2160,0.1738,0.0213
lasso,Lasso Regression,17.1888,532.0264,21.8674,-0.5131,0.2171,0.1808,0.0153
llar,Lasso Least Angle Regression,17.1888,532.0264,21.8674,-0.5131,0.2171,0.1808,0.0160
en,Elastic Net,17.2040,533.6467,21.8963,-0.5183,0.2174,0.1810,0.0193
ridge,Ridge Regression,17.2382,536.7740,21.9518,-0.5274,0.2180,0.1813,0.0153
lr,Linear Regression,17.2384,536.7924,21.9522,-0.5275,0.2180,0.1813,0.0207
lar,Least Angle Regression,17.2384,536.7924,21.9522,-0.5275,0.2180,0.1813,0.0147


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,12.7885,239.2203,15.4667,0.2541,0.1775,0.1454


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,14.8136,282.1541,16.7974,0.3519,0.1827,0.1677
1,11.9718,189.2485,13.7568,0.0510,0.1487,0.1312
2,20.9774,565.5287,23.7808,0.0685,0.2490,0.2256
3,25.4832,1372.5657,37.0482,-0.1933,0.2834,0.1865
4,14.0047,320.0152,17.8890,-0.1444,0.1616,0.1224
5,20.2900,572.2109,23.9209,-2.3863,0.2899,0.2896
6,9.1355,118.5874,10.8898,0.1680,0.1176,0.1027
7,14.2527,281.7387,16.7851,0.1655,0.1791,0.1595
8,16.8382,477.5959,21.8540,-0.4002,0.2838,0.2544


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,11.2486,242.5454,15.5739,0.2437,0.1811,0.1319


¡Proceso terminado!
LGBMRegressor(device='gpu', n_jobs=-1, random_state=123)
Proceso finalizado para Región de Magallanes y de la Antártica Chilena


El modelado para la Región de Magallanes presenta los peores resultados de todo el estudio, siendo un caso crítico de falla estructural. Aunque se seleccionó LightGBM como el mejor modelo, su desempeño es inaceptable: el $R^2$ promedio de validación cruzada es -0.32, lo que significa que el modelo es peor que usar el promedio simple y no tiene ninguna capacidad predictiva real. El análisis de folds revela un caos total: casi todos los folds presentan valores negativos o cercanos a cero, con colapsos catastróficos en el Fold 5 ($R^2$ -2.38) y el Fold 11 ($R^2$ -1.87). Esto confirma que la temporalidad de Magallanes (con inviernos muy largos sin incendios y veranos cortos pero intensos) es incompatible con la fragmentación de 15 folds, haciendo imposible que el modelo aprenda patrones estacionales coherentes.